# Non-pivot xslx functions 3

Created: 2026-10-17

Another try at parsing the column headers using a tree data structure.

The previous notebook in this series [Non-pivot xslx functions 2](non-pivot-xlsx-02.ipynb) worked through functionality for tables in the [December 2014 statistics for foreign residents in Japan](http://www.e-stat.go.jp/SG1/estat/List.do?lid=000001133760). This notebook is for adding this new functionality to the `excel` module. Here I use the tables for the [December 2024 statistics for foreign residents in Japan](http://www.e-stat.go.jp/SG1/estat/List.do?lid=000001462230) in addition to the earlier December 2014 dataset. The December 2024 dataset contains excel files that are in column-oriented format, excel files that are in the 2014 reporting format with some changes, and excel files that contain pivot tables.

## Install the necessary packages

In [ ]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install numpy
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl
%pip install treelib

## Install the newest version of estatjp

Repeat during development to import revisions.

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [1]:
url12120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315227&fileKind=0"
url12120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315228&fileKind=0"
url12120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315229&fileKind=0"
url12120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315230&fileKind=0"
url121203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315231&fileKind=0"
url121204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315232&fileKind=0"
url12120500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315233&fileKind=0"
url12120501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315234&fileKind=0"
url12120502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315235&fileKind=0"
url12120503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315236&fileKind=0"
url12120504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315237&fileKind=0"
url12120505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315238&fileKind=0"
url12120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315239&fileKind=0"
url12120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315240&fileKind=0"
url12120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315241&fileKind=0"
url12120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315242&fileKind=0"
url12120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315303&fileKind=0"
url12120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315304&fileKind=0"
url121207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000022401560&fileKind=0"

url20120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104290&fileKind=4"
url20120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104291&fileKind=0" # column-oriented
url20120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104292&fileKind=4"
url20120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104293&fileKind=0" # column-oriented
url201203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104294&fileKind=4"
url201204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104295&fileKind=4"
url201205 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104296&fileKind=4"
url20120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104297&fileKind=4"
url20120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104298&fileKind=4"
url20120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104299&fileKind=4"
url20120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104300&fileKind=4"
url20120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104301&fileKind=4"
url20120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104302&fileKind=4"
url20120701 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104303&fileKind=0"
url20120702 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104304&fileKind=0"
url2012new = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104608&fileKind=0"

# 14-12-01-1 国籍・地域別　在留資格（在留目的）別　在留外国人
url14120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226524&fileKind=0" # 14-12-01-01 - Dec 2014 by nationality/region and residence class.
url14120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226525&fileKind=0" # 14-12-01-02 - Dec 2014 by nationality/region and residence class.
url14120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226526&fileKind=0" # 14-12-02-01 - Dec 2014 by nationality/region age and sex.
url14120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226527&fileKind=0" # 14-12-02-01 - Dec 2014 by nationality/region age and sex.
url141203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226528&fileKind=0" # 14-12-03 - Dec 2014 by residence class, age, sex
url141204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226529&fileKind=0" # 14-12-04 - Dec 2014 by prefecture, nationality/region
url14120500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226530&fileKind=0" # 14-12-05-00 - Dec 2014 by prefecture, residence class. (total)
url14120501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226531&fileKind=0" # 14-12-05-01 - Dec 2014 by prefecture, residence class. (China)
url14120502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226612&fileKind=0" # 14-12-05-02 - Dec 2014 by prefecture, residence class. (Taiwan)
url14120503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226613&fileKind=0" # 14-12-05-03 - Dec 2014 by prefecture, residence class. (Korea, NK)
url14120504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226614&fileKind=0" # 14-12-05-04 - Dec 2014 by prefecture, residence class. (Phillipines)
url14120505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226615&fileKind=0" # 14-12-05-05 - Dec 2014 by prefecture, residence class. (Brazil)
url14120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226616&fileKind=0" # 14-12-06-00 - Dec 2014 by prefecture, age and sex. (total)
url14120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226617&fileKind=0" # 14-12-06-01 - Dec 2014 by prefecture, age and sex. (China)
url14120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226618&fileKind=0" # 14-12-06-02 - Dec 2014 by prefecture, age and sex. (Taiwan)
url14120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226619&fileKind=0" # 14-12-06-03 - Dec 2014 by prefecture, age and sex. (Korea, NK)
url14120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226620&fileKind=0" # 14-12-06-04 - Dec 2014 by prefecture, age and sex. (Phillipines)
url14120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226621&fileKind=0" # 14-12-06-05 - Dec 2014 by prefecture, age and sex. (Brazil)
url141207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000029226622&fileKind=0" # 14-12-07 - Dec 2014 by municipality, nationality/region

url21120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032212359&fileKind=4"
url21120102m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213254&fileKind=0"
url21120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213274&fileKind=4"
url21120202m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213275&fileKind=0"
url21120301 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213255&fileKind=0"
url21120302 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213256&fileKind=0"
url21120t1 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032213277&fileKind=0"

# worksheets that are in column-oriented format
url221201m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040068461&fileKind=0"
url22120102m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040248786&fileKind=0"
url221202m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040068660&fileKind=0"
url22120202m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040248848&fileKind=0"
# worksheets that are in reporting format
url221203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040068661&fileKind=0"
url221204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040068662&fileKind=0"
# worksheets that have pivot tables
url22120t1 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040068664&fileKind=0"

# worksheets that are in column-oriented format
url231201m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186952&fileKind=0"
url23120102m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040248882&fileKind=0"
url231202m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186953&fileKind=0"
url231203m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186954&fileKind=0"
url23120302m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040248883&fileKind=0"
# worksheets that have pivot tables
url23120t1 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186956&fileKind=0"
url23120t2 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186957&fileKind=0"
# worksheets that are in reporting format
url2312a = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186955&fileKind=0"

# worksheets that are in column-oriented format
url25120101m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472260&fileKind=0"
url25120102m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472261&fileKind=0"
url251202m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472262&fileKind=0"
url25120301m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472263&fileKind=0"
url25120302m = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472264&fileKind=0"
# worksheets that have pivot tables
url25120t1 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472265&fileKind=0"
url25120t2 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040472266&fileKind=0"
# worksheets that are in reporting format
url25120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040292366&fileKind=4"
url25120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040292367&fileKind=4"
url251202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040292368&fileKind=4"
url25120301 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040292369&fileKind=4"
url25120302 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040292370&fileKind=4"

## Import the module and download a file

In [2]:
from estatjp.xlsx import xlsx
from pathlib import Path

theurl = url12120101
dataset = "201212"
ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa") / dataset
if Path.exists(dataraw) == False:
    Path.mkdir(dataraw)
dataprocessed = ROOT_DIR / Path(".data/processed/isa") / dataset
if Path.exists(dataprocessed) == False:
    Path.mkdir(dataprocessed)
newdata = xlsx.download_to_folder(theurl, dataraw, overwrite=True, verbose=True)
print('source: ' + str(newdata))
if newdata != None:
    processeddata = dataprocessed / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Cache-Control': 'private', 'X-Cache': 'MISS', 'X-Content-Type-Options': 'nosniff', 'Accept-Ranges': 'bytes', 'X-Frame-Options': 'SAMEORIGIN', 'Date': 'Wed, 11 Nov 2026 00:39:04 GMT', 'X-Zen-Fury': '7b35bcf22ae5c00a056e2981d6443be86ceab02a', 'Content-Disposition': "attachment; filename*=UTF-8''12-12-01-1.xlsx", 'X-XSS-Protection': '1; mode=block', 'Server': 'ZENEDGE', 'Content-Encoding': 'gzip', 'X-Cache-Status': 'NOTCACHED', 'Strict-Transport-Security': 'max-age=0', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\201212\12-12-01-1.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\201212\12-12-01-1.csv


## Examine the worksheets and look for pivot tables or column-orientation

If there are pivot tables, use the pivot module. Otherwise use the excel module. This notebook is only for reporting workbooks and doesn't use analytical pivot workbooks.

File names of pivot and column-oriented workbooks end with the letter 'm'. Pivot workbooks contain a worksheet named 'PVT'. Column-oriented workbooks contain a worksheet with the same name as the file name. (I don't yet know if these features are used consistently.)

In [3]:
import re
import datetime as dt
import openpyxl

SHEETNAME_REGEX = re.compile(r"(?P<year>\d{2})-(?P<month>\d{2})-(?P<tableno>\d{1,2}(-[\da-zA-Z0-9_]+)?)")
wb = openpyxl.load_workbook(newdata, data_only=True)
snames = wb.sheetnames
print(snames)
has_pivot = False
has_pivot_cache_records = False
is_column_oriented = False
for ws in wb.worksheets:
    print('title: ' + ws.title)
    print('dimensions: ' + ws.dimensions)
    if ws.print_title_cols != None:
        print('print_title_cols: ' + ws.print_title_cols)
    if ws.print_title_rows != None:
        print('print_title_rows: ' + ws.print_title_rows)
    print('_pivots len: ' + str(len(ws._pivots)))
    if len(ws._pivots) > 0:
        has_pivot = True
        print('pivot_table.cache.records: ' + str(ws._pivots[0].cache.records))
        if ws._pivots[0].cache.records != None:
            has_pivot_cache_records = True
    print('merged_cells len: ' + str(len(ws.merged_cells.ranges)))
    # the year, month and table name are in ws.title. This is the name of the sheet if not a pivot table.
    mt = SHEETNAME_REGEX.match(ws.title)
    if mt != None:
        print('tableno: ' + mt.group('tableno'))
        is_column_oriented = mt.group('tableno').endswith('m')

dataformat = 'CTF'
if has_pivot == True:
    dataformat = 'PVT'
elif is_column_oriented == True:
    dataformat = 'COF'

print('Data format is ' + dataformat)


['12-12-01-1']
title: 12-12-01-1
dimensions: A2:AT302
print_title_cols: $A:$B
print_title_rows: $1:$6
_pivots len: 0
merged_cells len: 48
tableno: 01-1
Data format is CTF


### Use a [regular expression](https://docs.python.org/3/library/re.html) to identify and parse the title

Use an actual example to develop the regular expression. The group names 'cat01' and 'cat02' match those used by e-Stat's API service (See [Accessing the MOJ/ISA data via API calls](moj-isa-via-api.ipynb)).

In [4]:
import re
TITLE_REGEX = re.compile(r"第\d+表\S*\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<title_population>\w+)\s*(（(その\w+\s+)?(?P<sub_population>[\s\S]+)）)?")
CAT_LABEL_REGEX = re.compile(r"(?P<cat_label>\w+)（(?P<sub_cat_label>\w+)）")
title_str = "第１表　国籍・地域別　在留資格（在留目的）別　在留外国人" 
m= TITLE_REGEX.match(title_str)
print(m)
if m != None:
    print(m.group('cat01_label'))
    print(m.group('cat02_label'))
    print(m.group('title_population'))
    print(m.group('sub_population'))
    msub = CAT_LABEL_REGEX.match(m.group('cat02_label'))
    if msub != None:
        print(msub.group('cat_label'))
        print( msub.group('sub_cat_label'))
else:
    print('irregular title: ' + title_str)

<re.Match object; span=(0, 28), match='第１表\u3000国籍・地域別\u3000在留資格（在留目的）別\u3000在留外国人'>
国籍・地域
在留資格（在留目的）
在留外国人
None
在留資格
在留目的


### Look for title information

The title over the table usually contains information that is to be included in the dictionary entries to be passed to `pandas.from_dict` to create the data frame.

In [5]:
from openpyxl import Workbook
from openpyxl.cell.cell import Cell
from openpyxl.worksheet.cell_range import (MultiCellRange, CellRange)
from openpyxl.utils import (
    get_column_letter,
)

#TITLE_REGEX = re.compile(r"第\d+表\S*\s+(?P<cat01_label>\S+)別\s+(?P<cat02_label>\S+)別\s+(?P<title_population>\w+)(（(その\w+\s+)?(?P<sub_population>\S+)）)?")
ws = wb.active # There is only one worksheet in the current workbook
print(ws.print_title_rows)
print(ws.print_titles)
if ws._print_rows != None:
    print("min print row: " + str(ws._print_rows.min_row))
    print("max print row: " + str(ws._print_rows.max_row))
if ws._print_cols != None:
    print("min print column: " + str(ws._print_cols.min_col))
    print("max print column: " + str(ws._print_cols.max_col))

coord = ws.dimensions
stated_range = CellRange(coord)
stated_left_col = get_column_letter(stated_range.min_col)
print('Stated dimensions: ' + str(coord))
title_result = {}
# the year, month and table name are in ws.title. This is the name of the sheet.
mt = SHEETNAME_REGEX.match(ws.title)
if mt != None:
    yr = int("20" + mt.group('year'))
    month = int(mt.group('month'))
    sd = dt.date(yr, month, 1)
    statsdtstr = str(sd.year) + "年" + str(sd.month) + "月"
    tcstr = "20" + mt.group('year') + "00" + mt.group('month') + mt.group('month')
    title_result['table_name'] = ws.title.strip()
    title_result['time_code'] = int(tcstr)
    title_result['stats_date'] = statsdtstr

# look for the first row with the rightmost cell in the maximum column
bounds = openpyxl.utils.cell.range_boundaries(coord)
print(bounds)
stated_min_col = bounds[0]
stated_min_row = bounds[1]
stated_max_col = bounds[2]
stated_max_row = bounds[3]
found_max_col = stated_min_col

allowed_rows = 5
found_title_row = 0
rowriter = ws.iter_rows(min_row=stated_min_row, max_row=stated_min_row+allowed_rows, min_col = stated_min_col, max_col=stated_max_col)
for idx, row in enumerate(rowriter):
    # Parse until we get the title
    m = None
    for cell in row:
        if cell.value != None:
            m = TITLE_REGEX.match(cell.value)
            if m != None:
                found_title_row = cell.row
                title_result["title_population"] = m.group('title_population')
                if m.group('sub_population') != None:
                    title_result["sub_population"] = re.sub(r'[\s\n\u3000]' , '', m.group('sub_population'))
                else:
                    title_result["sub_population"] = '<NA>'
                title_result["cat01_label"] = m.group('cat01_label')
                title_result["cat02_label"] = m.group('cat02_label')
    if m != None:
        break
print('Found title on row ' + str(found_title_row))
print('Dictionary items for the dataframe: ')
print(title_result)


$1:$6
'12-12-01-1'!$1:$6,'12-12-01-1'!$A:$B
min print row: 1
max print row: 6
min print column: A
max print column: B
Stated dimensions: A2:AT302
(1, 2, 46, 302)
Found title on row 2
Dictionary items for the dataframe: 
{'table_name': '12-12-01-1', 'time_code': 2012001212, 'stats_date': '2012年12月', 'title_population': '在留外国人', 'sub_population': '<NA>', 'cat01_label': '国籍・地域', 'cat02_label': '在留資格（在留目的）'}


## Scan the worksheet to isolate tables

Worksheet dimensions are not reliable for determining the columns and rows of the table. Some worksheets have blank columns left and right, and some have blank rows below the table.

The [previous notebook](non-pivot-xlsx-02.ipynb) used tables from 2014 that have merged cell areas that cleanly demarked column headers. Tables from December 2020 do not cleanly use merged cells in the column headers. This notebook will try determining the table dimensions based on the minimum block that contains cells with either string or numerical values like in the previous notebook. It will then determine a subblock containing only numerical values.

For this workbook, the table and its parts are located as shown in the figure below. First the cell range that covers the entire table is determined by counting the maximum number of cells with values among the rows. This cell range becomes `valcellrange`. Next, determine the subrange that contains numerical values. This becomes `numcellrange`. The columns to the left of `numcellrange` become the stub column; those above become the dependent variable cell range `depvarheadrange`, from which the `crowntree` tree structure for parsing the column headers will be derived. 

The row indexes are parsed using a tree structure applied to the stub and stub row range.


<img src="images/TableRanges.gif" width="600">



In [6]:
from openpyxl import Workbook
from openpyxl.cell.cell import Cell
from openpyxl.cell.cell import MergedCell
from openpyxl.worksheet.cell_range import (MultiCellRange, CellRange)

valcellrange = None
numcellrange = None

rowiter = ws.iter_rows(min_row=found_title_row+1, max_row=stated_max_row, min_col = stated_min_col, max_col=stated_max_col)
#rows = openpyxl.utils.rows_from_range(coord)
for row in rowiter:
    hitcount = 0
    hitcellrange = None
    blankcellrange = None
    blankcount = 0
    max_val_col = 0
    ncellrange = None
    ncellcount = 0
    for idxcell, cell in enumerate(row):
        if (isinstance(cell, Cell) and cell.value != None):
            hitcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(hitcellrange, CellRange):
                hitcellrange = hitcellrange.union(thiscellrange)
            else:
                hitcellrange = thiscellrange
            if idxcell+1 > max_val_col:
                max_val_col = cell.column
                if max_val_col > found_max_col:
                    found_max_col = max_val_col
            if cell.data_type == 'n':
                ncellcount += 1
                if isinstance(ncellrange, CellRange):
                    ncellrange = ncellrange.union(thiscellrange)
                else:
                    ncellrange = thiscellrange
        elif (isinstance(cell, Cell) and cell.value == None) or isinstance(cell, MergedCell):
            blankcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(blankcellrange, CellRange):
                blankcellrange = blankcellrange.union(thiscellrange)
            else:
                blankcellrange = thiscellrange
    if isinstance(numcellrange, CellRange) == True and blankcount > hitcount and max_val_col < found_max_col:
        break
    if isinstance(valcellrange, CellRange) and isinstance(hitcellrange, CellRange):
        valcellrange = valcellrange.union(hitcellrange)
    elif isinstance(hitcellrange, CellRange):
        valcellrange = hitcellrange
    if isinstance(numcellrange, CellRange) and isinstance(ncellrange, CellRange):
        numcellrange = numcellrange.union(ncellrange)
    elif isinstance(hitcellrange, CellRange):
        numcellrange = ncellrange
    print(str(hitcount) + ' values out of ' + str(len(row)) + ' max column=' + str(max_val_col) + ' blanks: ' + str(blankcount))
    print(str(ncellcount) + ' numerical values out of ' + str(len(row)) + ' max column=' + str(max_val_col))

print('Value cell range: ' + str(valcellrange))
print('Numerical cell range: ' + str(numcellrange))

0 values out of 46 max column=0 blanks: 46
0 numerical values out of 46 max column=0
30 values out of 46 max column=45 blanks: 16
0 numerical values out of 46 max column=45
10 values out of 46 max column=40 blanks: 36
0 numerical values out of 46 max column=40
9 values out of 46 max column=39 blanks: 37
0 numerical values out of 46 max column=39
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 blanks: 2
43 numerical values out of 46 max column=45
44 values out of 46 max column=45 bl

### From numcellrange and valcellrange, derive column header range, row index range and stub cell range.

In [7]:
ncrsize = numcellrange.size
print(ncrsize)
colheaderrange = valcellrange.__copy__()
colheaderrange.shrink(bottom=ncrsize['rows'])
print('Column header range: ' + str(colheaderrange))

rowindexrange = valcellrange.__copy__()
rowindexrange.shrink(right=ncrsize['columns'])
print('Stub column range: ' + str(rowindexrange))
stubcellrange = colheaderrange.intersection(rowindexrange)
print('Stub cell range: ' + str(stubcellrange))
scrsize = stubcellrange.size
rowindexrange.shrink(top=scrsize['rows'])
colcount = rowindexrange.size['columns']
print('Stub row range: ' + str(rowindexrange))
depvarheadrange = colheaderrange.__copy__()
depvarheadrange.shrink(left=scrsize['columns'])
print('Dependent column header range: ' + str(depvarheadrange))


{'columns': 43, 'rows': 200}
Column header range: A4:AS6
Stub column range: A4:B206
Stub cell range: A4:B6
Stub row range: A7:B206
Dependent column header range: C4:AS6


### Carry functions to parse and duplicate header strings that span multiple columns or multiple rows 

`url20120101` presents multiple patterns of column headers that have significant manual character. 

- Column E centers its value by placing it in the middle cell. 
- Column G centers its value by placing it in the top cell and merging this with the two cells below it. 
- Column AG starts a set of decked headers with a value in the top cell that is then merged over many columns to its right. The middle cell of Column AG has a value that is centered by merging with the bottom cell. 
- Column AM's top cell is a MergedCell. Its value is from the top cell of column AG. Column AM's middle cell has is type Cell with a value that extends into column AN by merging. Column AM's botton cell contains a value.
- Column AN's top cell continues the MergedCell from column AG. Its middle cell extends by MergedCell the value from column AM. Its bottom cell contains an independent value.
- Column AO's top cell continues the MergedCell from column AG. Its middle cell contains the first part of an independent value with the last part in the bottom cell.


| ![Table 20-12-01-1 columns D-F](images/Table20-12-01-1-colsD-F.gif) | ![Table 20-12-01-1 columns AM-AO](images/Table20-12-01-1-colsAM-AO.gif) |
|-------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| Table 20-12-01-1 columns D-F                                                 | Table 20-12-01-1 columns AM-AO                                               |

| Column E | Column G |  | Column AG (Not shown) | Column AM | Column AN | Column AO |
|----------|----------|--|-----------------------|-----------|-----------|-----------|
| Cell value=None | Cell value='高度専門職1号イ' | | Cell value='特定活動' | MergedCell value=None | MergedCell value=None | MergedCell value=None |
| Cell value='宗教' | MergedCell value=None | | Cell value='計' | Cell value='アマスポーツ選手' | MergedCell value=None | Cell value='インターン' |
| Cell value=None | MergedCell value=None | | MergedCell value=None | Cell value='本人' | Cell value='家族' | Cell value='シップ' |

## Build the stub column tree

Use package [`treelib`](https://treelib.readthedocs.io/en/latest/modules.html).



### data_to_leaves 

Utility function adapted from [treelib.tree.paths_to_leaves](https://treelib.readthedocs.io/en/latest/_modules/treelib/tree.html#Tree.paths_to_leaves)

In [8]:
def paths_to_leaves_data(atree, field=None, makeuniform=True, noneform=None):
    """
    Use this function to get a data item allowing to go from the root
    nodes to each leaf.

    :return: a list of data items, root being not omitted.

    """
    res = {}
    templatedict = {} # the dictionaries ptol must be equal length, so create a template
    if makeuniform == True:
        for leaf in atree.leaves():
            nids = [nid for nid in atree.rsearch(leaf.identifier)][::-1]
            # build templatedict
            for nid in nids:
                fdict = atree[nid].data[field]
                d = { k: noneform for k in list(fdict)}
                templatedict.update(d)

    for leaf in atree.leaves():
        nids = [nid for nid in atree.rsearch(leaf.identifier)][::-1]
        ptol = templatedict.copy()
        lid = leaf.identifier
        # sort by nid to put into column order
        for nid in nids:
            ptol.update(atree[nid].data[field])
        res.update({lid: ptol})

    return res


### Parse the column headers

`dfdict` in the `data` parameter of `tree.create_note` will be part of the dict row when creating the `pandas.DataFrame`.

See [Working with Custom Data](https://treelib.readthedocs.io/en/latest/) and [Defining an Iterable Object Class in Python](https://www.bing.com/search?q=python+make+class+iterable&form=ANNTH1&refig=6adde814cf234b18bbd06fe751512c37&pc=LCTS)

In [9]:
from types import SimpleNamespace

class DFDict(dict):
    def __init__(self, *arg, **kw):
        super(DFDict, self).__init__(*arg, **kw)


### Add the dictionaries `dfdict` that go into the column-oriented dataframe.

Put the table title parse results in the stub (cat01) branch.

In [10]:
from treelib import Tree, Node

stubtree = Tree()
# put title derived parameters in the root
stubtree.create_node(title_result['table_name'], 'root',data={'dfdict':{}})
# create children for stub column (cat01) and dependent columns (cat02)
stubtree.create_node(title_result["cat01_label"], 'cat01', parent='root', 
                 data={'value': '<NV>', 'dfdict':{
                                'table_name': title_result['table_name'], 
                                'time_code': title_result['time_code'],
                                'stats_date': title_result['stats_date'],
                                'title_population': title_result["title_population"],
                                'sub_population': title_result["sub_population"],
                                'cat01': title_result["cat01_label"] }})



Node(tag=国籍・地域, identifier=cat01, data={'value': '<NV>', 'dfdict': {'table_name': '12-12-01-1', 'time_code': 2012001212, 'stats_date': '2012年12月', 'title_population': '在留外国人', 'sub_population': '<NA>', 'cat01': '国籍・地域'}})

### Build dfdict entries for possible hierachy in stub column

Worksheet tables may have hierarchical stub columns. For instance, there may be a list of countries in column B with regions in column A. These may or may not be labeled as such. If labeled, use the label. If not, use 階級0, 階級1, etc to label the levels.

In [11]:
# create stub level dict
stubdict = {}

previousrow = [None] * stubcellrange.size['columns']
for idxrow, row in enumerate(ws.iter_rows(min_col=stubcellrange.min_col, 
                                          min_row=stubcellrange.min_row, 
                                          max_col=stubcellrange.max_col, 
                                          max_row=stubcellrange.max_row)):
    for idxcell, cell in enumerate(row):
        # some tests for "merged cells" that may not be in the <mergeCells> element
        # this first test is for horizontal alignment set to be centered in a selected range
        isCenterContinuous = cell.alignment.horizontal == "centerContinuous"
        hasValue = cell.value != None # mimics merge by checking for no value
        # this second is for cells merged by selecting "Merge cells"
        inMergedCells = ws.merged_cells.__contains__(cell.coordinate)
        isMergeHead = hasValue == True and (isCenterContinuous == True or inMergedCells)
        isMergedCell = hasValue == False and (isCenterContinuous == True or inMergedCells)
    
        if cell.value == None:
            val = str('階級' + str(idxcell))
        else:
            val = re.sub(r'[\s\n\u3000]' , '', cell.value)

        stubdict.update({'cat01lvl' + str(idxcell) + '_label': val})
      
  
stubdict

{'cat01lvl0_label': '階級0', 'cat01lvl1_label': '階級1'}

Row by row, check the stub header for hierarchically labeled rows. Add these to the stub tree.

In [12]:
# level by level transfer children of MergedCells to Cell preceeding MergeCell run
max_depth = max(stubtree.level(node.identifier)
              for node in stubtree.all_nodes())
toRemove = []
# for cells that have only one child, append child's value to self and remove child with link_past_node
for i in range(2,max_depth+1):
    level_nodes = [node for node in stubtree.all_nodes() if stubtree.level(node.identifier) == i]
    for node in level_nodes:
        kids = stubtree.children(node.identifier)
        thisval = '<NA>'
        thisdata = node.data.copy()
        thisdfdict = thisdata['dfdict'].copy()
        if 'value' in thisdata:  
            thisval = thisdata['value']
        if len(kids) == 0 and thisval == '<NA>':
            if stubcellrange.__contains__(cell.coordinate) == False:
                stubtree.remove_node(node.identifier)
        if len(kids) == 1:
            kidval = kids[0].data['value']
            newval = '<NA>'
            if thisval == '<NA>' and kidval == '<NA>':
                newval = '<NA>'
            elif thisval != '<NA>' and kidval == '<NA>':
                newval = thisval
            elif kidval != '<NA>' and thisval == '<NA>':
                newval = kidval
            else:
                newval = thisval + kidval
            thisdata['value'] = newval
            thesekeys = thisdfdict.keys()
            thesevalues = thisdfdict.values()
            stubtree.update_node(node.identifier, 
                             tag=node.identifier + ' ' + newval)
            stubtree.link_past_node(kids[0].identifier)

stubtree.show(sorting=False)
stubleaves = paths_to_leaves_data(stubtree, field='dfdict', makeuniform=False)
stubleaves

12-12-01-1
└── 国籍・地域



{'cat01': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域'}}

In [13]:
# The 1st item is the parent of all first column row labels
stubleaves = paths_to_leaves_data(stubtree, field='dfdict')
stubparent = list(stubleaves)[0]
# Iterate the rows and in each row iterate the stub columns
row_head = []
carriednodes = [None] * colcount
# the stub column heads are for cat01. The rest are for cat02. 
cat01_levels = ['cat01lvl' + str(n) for n in range(colcount)]
lvl0node = stubparent
for idxrow, row in enumerate(ws.iter_rows(min_row=rowindexrange.min_row, 
                                          max_row=rowindexrange.max_row, 
                                          min_col=rowindexrange.min_col,
                                          max_col=rowindexrange.max_col)):
    attachnode = lvl0node
    for idxcell, c in enumerate(row):
        if c.value == None:
            val = '<NA>'
        else:
            val = re.sub(r'[\s\n\u3000]' , '', c.value)
        dfdictentry = {'cat01lvl' + str(idxcell): val}
        dfdictentry.update(stubdict)
        stubtree.create_node(c.coordinate + val, 
                        c.coordinate, 
                        parent=attachnode, 
                        data={'value': val, 'dfdict': dfdictentry})
        attachnode = c.coordinate
    
stubtree.show(sorting=False)
stubleaves = paths_to_leaves_data(stubtree, field='dfdict', makeuniform=False)
stubleaves

12-12-01-1
└── 国籍・地域
    ├── A7総数
    │   └── B7<NA>
    ├── A8アジア
    │   └── B8<NA>
    ├── A9<NA>
    │   └── B9アフガニスタン
    ├── A10<NA>
    │   └── B10アラブ首長国連邦
    ├── A11<NA>
    │   └── B11ミャンマー
    ├── A12<NA>
    │   └── B12バーレーン
    ├── A13<NA>
    │   └── B13ブータン
    ├── A14<NA>
    │   └── B14バングラデシュ
    ├── A15<NA>
    │   └── B15ブルネイ
    ├── A16<NA>
    │   └── B16カンボジア
    ├── A17<NA>
    │   └── B17スリランカ
    ├── A18<NA>
    │   └── B18中国
    ├── A19<NA>
    │   └── B19台湾
    ├── A20<NA>
    │   └── B20キプロス
    ├── A21<NA>
    │   └── B21東ティモール
    ├── A22<NA>
    │   └── B22インド
    ├── A23<NA>
    │   └── B23インドネシア
    ├── A24<NA>
    │   └── B24イラン
    ├── A25<NA>
    │   └── B25イラク
    ├── A26<NA>
    │   └── B26イスラエル
    ├── A27<NA>
    │   └── B27ヨルダン
    ├── A28<NA>
    │   └── B28韓国・朝鮮
    ├── A29<NA>
    │   └── B29クウェート
    ├── A30<NA>
    │   └── B30ラオス
    ├── A31<NA>
    │   └── B31レバノン
    ├── A32<NA>
    │   └── B32マレーシア
    ├── A33<NA>
    │   └── B33モンゴル
  

{'B7': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B8': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B9': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '<NA>',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': 'アフガニスタン'},
 'B10': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lv

### Pull row labels under respective higher levels

In [14]:
# level by level transfer children of MergedCells to Cell preceeding MergeCell run
max_depth = max(stubtree.level(node.identifier)
              for node in stubtree.all_nodes())
toRemove = []
for i in range(1,max_depth):
    accumhead = None 
    level_nodes = [node for node in stubtree.all_nodes() if stubtree.level(node.identifier) == i]
    for node in level_nodes:
        if node.data['value'] != '<NA>':
            accumhead = node.identifier
        else:
            for child in stubtree.children(node.identifier):
                stubtree.move_node(child.identifier, accumhead)
            toRemove.append(node.identifier)

for n in toRemove:
    stubtree.remove_node(n)

stubtree.show(sorting=False)
stubleaves = paths_to_leaves_data(stubtree, field='dfdict', makeuniform=False)
stubleaves

12-12-01-1
└── 国籍・地域
    ├── A7総数
    │   └── B7<NA>
    ├── A8アジア
    │   ├── B8<NA>
    │   ├── B9アフガニスタン
    │   ├── B10アラブ首長国連邦
    │   ├── B11ミャンマー
    │   ├── B12バーレーン
    │   ├── B13ブータン
    │   ├── B14バングラデシュ
    │   ├── B15ブルネイ
    │   ├── B16カンボジア
    │   ├── B17スリランカ
    │   ├── B18中国
    │   ├── B19台湾
    │   ├── B20キプロス
    │   ├── B21東ティモール
    │   ├── B22インド
    │   ├── B23インドネシア
    │   ├── B24イラン
    │   ├── B25イラク
    │   ├── B26イスラエル
    │   ├── B27ヨルダン
    │   ├── B28韓国・朝鮮
    │   ├── B29クウェート
    │   ├── B30ラオス
    │   ├── B31レバノン
    │   ├── B32マレーシア
    │   ├── B33モンゴル
    │   ├── B34オマーン
    │   ├── B35モルディブ
    │   ├── B36ネパール
    │   ├── B37パキスタン
    │   ├── B38フィリピン
    │   ├── B39カタール
    │   ├── B40サウジアラビア
    │   ├── B41シリア
    │   ├── B42シンガポール
    │   ├── B43タイ
    │   ├── B44トルコ
    │   ├── B45ベトナム
    │   ├── B46イエメン
    │   └── B47パレスチナ
    ├── A48ヨーロッパ
    │   ├── B48<NA>
    │   ├── B49アルバニア
    │   ├── B50オーストリア
    │   ├── B51ベルギー
    │   ├── B52ブ

{'B7': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B8': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B9': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': 'アフガニスタン'},
 'B10': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl

### Create tree for dependent column headers and parse headers

In [15]:
import re
# put units in the crown tree because there is likelihood that this will
# change per column.
UNITS_REGEX = re.compile(r"(?P<units>人|率)\Z")
titlepop = title_result["title_population"]
unitsmatch = UNITS_REGEX.search(titlepop)
unitsdict = {}
if unitsmatch != None:
    unitsdict = {'units': unitsmatch.group('units')}

# create the tree branch for the dependent columns (cat02)
treecrown = Tree()
treecrown.create_node('crowntree', 
                'root',
                data={'value': '<NA>', 'dfdict': unitsdict})
treecrown.create_node(title_result["cat02_label"], 'cat02', parent='root', 
                 data={'value': '<NA>', 'dfdict':{'cat02': title_result["cat02_label"]}})
# scan column headers by row
bnds = depvarheadrange.bounds
previousrow = [None] * depvarheadrange.size['columns']
for idxrow, crownrow in enumerate(ws.iter_rows(
                                        min_col=depvarheadrange.min_col, 
                                        min_row=depvarheadrange.min_row, 
                                        max_col=depvarheadrange.max_col, 
                                        max_row=depvarheadrange.max_row)):
    for idxcell, cell in enumerate(crownrow):
        # some tests for "merged cells" that may not be in the <mergeCells> element
        # this first test is for horizontal alignment set to be centered in a selected range
        isCenterContinuous = cell.alignment.horizontal == "centerContinuous"
        hasValue = cell.value != None # mimics merge by checking for no value
        # this second is for cells merged by selecting "Merge cells"
        inMergedCells = ws.merged_cells.__contains__(cell.coordinate)
        isMergeHead = hasValue == True and (isCenterContinuous == True or inMergedCells)
        isMergedCell = hasValue == False and (isCenterContinuous == True or inMergedCells)
    
        if cell.value == None:
            val = '<NA>'
        else:
            val = re.sub(r'[\s\n\u3000]' , '', cell.value)

        prnt = None
        dfdictentry = {}
        prnt = 'cat02'
 
        if previousrow[idxcell] != None:
            prnt = previousrow[idxcell]
        
        if isMergeHead:
            treecrown.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                             data={'MergeType': 'MergeHead', 'datatype': 'Cell', 'value': val, 'dfdict': dfdictentry})
            previousrow[idxcell] = cell.coordinate
        elif isMergedCell:
            treecrown.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                             data={'MergeType': 'MergedCell', 'datatype': 'MergedCell', 'value': val, 'dfdict': dfdictentry})
            previousrow[idxcell] = cell.coordinate
        elif isinstance(cell, Cell):
            treecrown.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                             data={'MergeType': 'Cell', 'datatype': 'Cell', 'value': val, 'dfdict': dfdictentry})
            previousrow[idxcell] = cell.coordinate
    

treecrown.show(sorting=False)
crownleaves = paths_to_leaves_data(treecrown, field='dfdict',makeuniform=False)
crownleaves

crowntree
└── 在留資格（在留目的）
    ├── C4 総数
    │   └── C5 <NA>
    │       └── C6 <NA>
    ├── D4 教授
    │   └── D5 <NA>
    │       └── D6 <NA>
    ├── E4 芸術
    │   └── E5 <NA>
    │       └── E6 <NA>
    ├── F4 宗教
    │   └── F5 <NA>
    │       └── F6 <NA>
    ├── G4 報道
    │   └── G5 <NA>
    │       └── G6 <NA>
    ├── H4 投資・経営
    │   └── H5 <NA>
    │       └── H6 <NA>
    ├── I4 法律・会計業務
    │   └── I5 <NA>
    │       └── I6 <NA>
    ├── J4 医療
    │   └── J5 <NA>
    │       └── J6 <NA>
    ├── K4 研究
    │   └── K5 <NA>
    │       └── K6 <NA>
    ├── L4 教育
    │   └── L5 <NA>
    │       └── L6 <NA>
    ├── M4 技術
    │   └── M5 <NA>
    │       └── M6 <NA>
    ├── N4 人文知識・国際業務
    │   └── N5 <NA>
    │       └── N6 <NA>
    ├── O4 企業内転勤
    │   └── O5 <NA>
    │       └── O6 <NA>
    ├── P4 興行
    │   └── P5 <NA>
    │       └── P6 <NA>
    ├── Q4 技能
    │   └── Q5 <NA>
    │       └── Q6 <NA>
    ├── R4 技能実習１号イ
    │   └── R5 <NA>
    │       └── R6 <NA>
    ├── S4 技能実習１号ロ
    │

{'C6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'D6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'E6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'F6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'G6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'H6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'I6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'J6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'K6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'L6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'M6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'N6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'O6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'P6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'Q6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'R6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'S6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'T6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'U6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'V6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'W6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'X6': {'units': '人', 'cat02': '在留

The above tree is simply columnar subtrees under either the stub (cat01) category or the dependent (cat02) category.

### Pull columns under decked headers under their respective merge heads

level by level transfer children of MergedCells to Cell preceeding MergeCell run

In [16]:
# level by level transfer children of MergedCells to Cell preceeding MergeCell run
max_depth = max(treecrown.level(node.identifier)
              for node in treecrown.all_nodes())
toRemove = []
for i in range(2,max_depth):
    mergeHead = None # when scanning left to right, if a node is a MergeHead, copy its id here. Then move the children of following MergedCells to it.
    level_nodes = [node for node in treecrown.all_nodes() if treecrown.level(node.identifier) == i]
    for node in level_nodes:
        if node.data['MergeType'] == 'MergeHead':
            mergeHead = node.identifier
        elif node.data['MergeType'] == 'Cell':
            mergeHead = None
        elif node.data['MergeType'] == 'MergedCell':
            if mergeHead != None:
                for child in treecrown.children(node.identifier):
                    treecrown.move_node(child.identifier, mergeHead)
                toRemove.append(node.identifier)
for n in toRemove:
    treecrown.remove_node(n)

treecrown.show(sorting=False)
crownleaves = paths_to_leaves_data(treecrown, field='dfdict',makeuniform=True, noneform='<NA>')
crownleaves


crowntree
└── 在留資格（在留目的）
    ├── C4 総数
    │   └── C5 <NA>
    │       └── C6 <NA>
    ├── D4 教授
    │   └── D5 <NA>
    │       └── D6 <NA>
    ├── E4 芸術
    │   └── E5 <NA>
    │       └── E6 <NA>
    ├── F4 宗教
    │   └── F5 <NA>
    │       └── F6 <NA>
    ├── G4 報道
    │   └── G5 <NA>
    │       └── G6 <NA>
    ├── H4 投資・経営
    │   └── H5 <NA>
    │       └── H6 <NA>
    ├── I4 法律・会計業務
    │   └── I5 <NA>
    │       └── I6 <NA>
    ├── J4 医療
    │   └── J5 <NA>
    │       └── J6 <NA>
    ├── K4 研究
    │   └── K5 <NA>
    │       └── K6 <NA>
    ├── L4 教育
    │   └── L5 <NA>
    │       └── L6 <NA>
    ├── M4 技術
    │   └── M5 <NA>
    │       └── M6 <NA>
    ├── N4 人文知識・国際業務
    │   └── N5 <NA>
    │       └── N6 <NA>
    ├── O4 企業内転勤
    │   └── O5 <NA>
    │       └── O6 <NA>
    ├── P4 興行
    │   └── P5 <NA>
    │       └── P6 <NA>
    ├── Q4 技能
    │   └── Q5 <NA>
    │       └── Q6 <NA>
    ├── R4 技能実習１号イ
    │   └── R5 <NA>
    │       └── R6 <NA>
    ├── S4 技能実習１号ロ
    │

{'AO4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'AP4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'AQ4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'AR4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'AS4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'C6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'D6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'E6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'F6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'G6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'H6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'I6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'J6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'K6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'L6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'M6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'N6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'O6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'P6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'Q6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'R6': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'S6': {'units': '人', 'cat02'

This leaves a set of columns with empty cells. Link past the empties and remove them.

In [17]:
# make a deep copy for developing this section
#duptree = Tree(treecrown.subtree(treecrown.root), deep=True)
#dupleaves = paths_to_leaves_data(duptree, field='dfdict',makeuniform=True, noneform='<NA>')
#dupleaves
# for cells that have only one child, append child's value to self and remove child with link_past_node
toRemove = []
for i in range(2,max_depth+1):
    level_nodes = [node for node in treecrown.all_nodes() if treecrown.level(node.identifier) == i]
    for node in level_nodes:
        kids = treecrown.children(node.identifier)
        prnt = treecrown.parent(node.identifier)
        thisval = '<NA>'
        thisdata = node.data.copy()
        thisdfdict = thisdata['dfdict'].copy()
        if 'value' in thisdata:  
            thisval = thisdata['value']
        if len(kids) == 0 and len(treecrown.siblings(node.identifier)) == 0 and thisval == '<NA>':
            toRemove.append(node.identifier)
        if len(kids) == 1:
            kidval = kids[0].data['value']
            newval = '<NA>'
            if thisval == '<NA>' and kidval == '<NA>':
                newval = '<NA>'
            elif thisval != '<NA>' and kidval == '<NA>':
                newval = thisval
            elif kidval != '<NA>' and thisval == '<NA>':
                newval = kidval
            else:
                newval = thisval + kidval
            node.data.update({'value': newval})
            treecrown.update_node(node.identifier, 
                             tag=node.identifier + ' ' + newval)
            treecrown.link_past_node(kids[0].identifier)
for n in toRemove:
    treecrown.remove_node(n)

treecrown.show(sorting=False)
crownleaves = paths_to_leaves_data(treecrown, field='dfdict',makeuniform=True, noneform='<NA>')
crownleaves

crowntree
└── 在留資格（在留目的）
    ├── C4 総数
    ├── D4 教授
    ├── E4 芸術
    ├── F4 宗教
    ├── G4 報道
    ├── H4 投資・経営
    ├── I4 法律・会計業務
    ├── J4 医療
    ├── K4 研究
    ├── L4 教育
    ├── M4 技術
    ├── N4 人文知識・国際業務
    ├── O4 企業内転勤
    ├── P4 興行
    ├── Q4 技能
    ├── R4 技能実習１号イ
    ├── S4 技能実習１号ロ
    ├── T4 技能実習２号イ
    ├── U4 技能実習２号ロ
    ├── V4 文化活動
    ├── W4 留学
    ├── X4 研修
    ├── Y4 家族滞在
    ├── Z4 特定活動
    │   ├── Z5 計
    │   ├── AA5 特定研究等及び情報処理
    │   │   ├── AA6 本人
    │   │   └── AB6 家族
    │   ├── AC5 家事使用人
    │   ├── AD5 ワーキング・ホリデー
    │   ├── AE5 アマスポーツ選手
    │   │   ├── AE6 本人
    │   │   └── AF6 家族
    │   ├── AG5 インターンシップ
    │   ├── AH5 ＥＰＡ対象者
    │   │   ├── AH6 本人
    │   │   └── AI6 家族
    │   ├── AJ5 医療滞在・同伴者
    │   ├── AK5 高度人材
    │   │   ├── AK6 本人
    │   │   ├── AL6 家族
    │   │   └── AM6 家事使用人
    │   └── AN5 その他
    │       ├── AN6 <NA>
    │       ├── AO6 <NA>
    │       ├── AP6 <NA>
    │       ├── AQ6 <NA>
    │       ├── AR6 <NA>
    │       └── AS6 <NA>
  

{'C4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'D4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'E4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'F4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'G4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'H4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'I4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'J4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'K4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'L4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'M4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'N4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'O4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'P4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'Q4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'R4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'S4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'T4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'U4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'V4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'W4': {'units': '人', 'cat02': '在留資格（在留目的）'},
 'X4': {'units': '人', 'cat02': '在留

Now put the rest of dfdict entries in the nodes

In [18]:
for i in range(2,max_depth+1):
    level_nodes = [node for node in treecrown.all_nodes() if treecrown.level(node.identifier) == i]
    for node in level_nodes:
        thisdata = node.data.copy()
        thisdfdict = thisdata['dfdict'].copy()
        thisdfdict.update({'cat02lvl' + str(i-2): thisdata['value']})
        node.data.update({'dfdict': thisdfdict})

treecrown.show(sorting=False)
crownleaves = paths_to_leaves_data(treecrown, field='dfdict',makeuniform=True, noneform='<NA>')
crownleaves

crowntree
└── 在留資格（在留目的）
    ├── C4 総数
    ├── D4 教授
    ├── E4 芸術
    ├── F4 宗教
    ├── G4 報道
    ├── H4 投資・経営
    ├── I4 法律・会計業務
    ├── J4 医療
    ├── K4 研究
    ├── L4 教育
    ├── M4 技術
    ├── N4 人文知識・国際業務
    ├── O4 企業内転勤
    ├── P4 興行
    ├── Q4 技能
    ├── R4 技能実習１号イ
    ├── S4 技能実習１号ロ
    ├── T4 技能実習２号イ
    ├── U4 技能実習２号ロ
    ├── V4 文化活動
    ├── W4 留学
    ├── X4 研修
    ├── Y4 家族滞在
    ├── Z4 特定活動
    │   ├── Z5 計
    │   ├── AA5 特定研究等及び情報処理
    │   │   ├── AA6 本人
    │   │   └── AB6 家族
    │   ├── AC5 家事使用人
    │   ├── AD5 ワーキング・ホリデー
    │   ├── AE5 アマスポーツ選手
    │   │   ├── AE6 本人
    │   │   └── AF6 家族
    │   ├── AG5 インターンシップ
    │   ├── AH5 ＥＰＡ対象者
    │   │   ├── AH6 本人
    │   │   └── AI6 家族
    │   ├── AJ5 医療滞在・同伴者
    │   ├── AK5 高度人材
    │   │   ├── AK6 本人
    │   │   ├── AL6 家族
    │   │   └── AM6 家事使用人
    │   └── AN5 その他
    │       ├── AN6 <NA>
    │       ├── AO6 <NA>
    │       ├── AP6 <NA>
    │       ├── AQ6 <NA>
    │       ├── AR6 <NA>
    │       └── AS6 <NA>
  

{'C4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '総数',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'D4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '教授',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'E4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '芸術',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'F4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '宗教',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'G4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '報道',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'H4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '投資・経営',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'I4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '法律・会計業務',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'J4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '医療',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 'K4': {'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '研究',
  'c

The leaves need to be sorted by column. Use method in [Stack Overflow](https://stackoverflow.com/questions/11399384/extract-all-keys-from-a-list-of-dictionaries). And comparison method also in [Stack Overflow](https://stackoverflow.com/questions/5213033/sort-a-list-of-lists-with-a-custom-compare-function).

In [19]:
from functools import cmp_to_key
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string

def compare_columns(coord1, coord2):
    collet1, row1 = coordinate_from_string(coord1)
    collet2, row2 = coordinate_from_string(coord2)
    idx1 = column_index_from_string(collet1)
    idx2 = column_index_from_string(collet2)
    return idx1-idx2

def compare_rows(coord1, coord2):
    collet1, row1 = coordinate_from_string(coord1)
    collet2, row2 = coordinate_from_string(coord2)
    return row1-row2

crown_keys = list(crownleaves)
print(crown_keys)
# Source - https://stackoverflow.com/a/57003713
# Posted by Lars Blumberg, modified by community. See post 'Timeline' for change history
# Retrieved 2026-11-07, License - CC BY-SA 4.0
sortedcrownkeys = sorted(crown_keys, key=cmp_to_key(compare_columns))
print(sortedcrownkeys)
# from https://stackoverflow.com/questions/5925731/reorder-dictionary-in-python-according-to-a-list-of-values
column_leaves = {k: crownleaves[k] for k in sortedcrownkeys}
column_leaves

row_keys = list(stubleaves)
print(row_keys)
sortedrowkeys = sorted(row_keys, key=cmp_to_key(compare_rows))
print(sortedrowkeys)
row_leaves = {k: stubleaves[k] for k in sortedrowkeys}
row_leaves

['C4', 'D4', 'E4', 'F4', 'G4', 'H4', 'I4', 'J4', 'K4', 'L4', 'M4', 'N4', 'O4', 'P4', 'Q4', 'R4', 'S4', 'T4', 'U4', 'V4', 'W4', 'X4', 'Y4', 'AO4', 'AP4', 'AQ4', 'AR4', 'AS4', 'Z5', 'AC5', 'AD5', 'AG5', 'AJ5', 'AA6', 'AB6', 'AE6', 'AF6', 'AH6', 'AI6', 'AK6', 'AL6', 'AM6', 'AN6', 'AO6', 'AP6', 'AQ6', 'AR6', 'AS6']
['C4', 'D4', 'E4', 'F4', 'G4', 'H4', 'I4', 'J4', 'K4', 'L4', 'M4', 'N4', 'O4', 'P4', 'Q4', 'R4', 'S4', 'T4', 'U4', 'V4', 'W4', 'X4', 'Y4', 'Z5', 'AA6', 'AB6', 'AC5', 'AD5', 'AE6', 'AF6', 'AG5', 'AH6', 'AI6', 'AJ5', 'AK6', 'AL6', 'AM6', 'AN6', 'AO4', 'AO6', 'AP4', 'AP6', 'AQ4', 'AQ6', 'AR4', 'AR6', 'AS4', 'AS6']
['B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 'B33', 'B34', 'B35', 'B36', 'B37', 'B38', 'B39', 'B40', 'B41', 'B42', 'B43', 'B44', 'B45', 'B46', 'B47', 'B48', 'B49', 'B50', 'B51', 'B52', 'B53', 'B54', 'B55', 'B56', 'B57', 'B58', 'B59', 'B60'

{'B7': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B8': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>'},
 'B9': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': 'アフガニスタン'},
 'B10': {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl

`dfdict` in the `data` parameter of `tree.create_note` will be part of the dict row when creating the `pandas.DataFrame`.

### Scan the body to make a list of row dictionaries that will be used to create the `pandas.dataframe`

In [20]:
rows = []
for idxrow, row in enumerate(ws.iter_rows(
        min_row = numcellrange.min_row,
        max_row = numcellrange.max_row,
        min_col = numcellrange.min_col,
        max_col = numcellrange.max_col )):
    rowdict = row_leaves[sortedrowkeys[idxrow]]
    for idxcol, cell in enumerate(row):
        celldict = {}
        celldict.update(rowdict)
        coldict = column_leaves[sortedcrownkeys[idxcol]]
        celldict.update(coldict)
        celldict['value'] = cell.value
        rows.append(celldict)

rows


[{'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>',
  'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '総数',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>',
  'value': 2033656},
 {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'cat01lvl1_label': '階級1',
  'cat01lvl1': '<NA>',
  'units': '人',
  'cat02': '在留資格（在留目的）',
  'cat02lvl0': '教授',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>',
  'value': 7787},
 {'table_name': '12-12-01-1',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0',
  'c

In [ ]:
import pandas as pd
import csv

cols = rows[0].keys
df = pd.DataFrame(data=rows, dtype=str)

df.to_csv(processeddata, index=False, quoting=csv.QUOTE_ALL)
df